In [1]:
pip install torch torchvision opencv-python pillow numpy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/530.6 MB 230.1 kB/s eta 0:38:05^C
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/530.6 MB 230.1 kB/s eta 0:38:05
ERROR: Operation cancelled by user

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
print("CI/CD")

CI/CD


In [2]:
print("JENKINS FINAL VERSION")

JENKINS FINAL VERSION


In [8]:
pip install ultralytics


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.0 requires FuzzyTM>=0.4.0, which is not installed.



  Obtaining dependency information for ultralytics from https://files.pythonhosted.org/packages/ef/32/146220e54bd87bb48faeb402ef7b2968ad2f0edd7d799787abffeb1d0724/ultralytics-8.3.225-py3-none-any.whl.metadata
  Obtaining dependency information for polars from https://files.pythonhosted.org/packages/9f/4c/21a227b722534404241c2a76beceb7463469d50c775a227fc5c209eb8adc/polars-1.35.1-py3-none-any.whl.metadata
  Obtaining dependency information for ultralytics-thop>=2.0.18 from https://files.pythonhosted.org/packages/7f/c7/fb42228bb05473d248c110218ffb8b1ad2f76728ed8699856e5af21112ad/ultralytics_thop-2.0.18-py3-none-any.whl.metadata
INFO: pip is looking at multiple versions of scipy to determine which version is compatible with other requirements. This could take a while.
  Obtaining dependency information for scipy>=1.4.1 from https://files.pythonhosted.org/packages/f1/d0/22ec7036ba0b0a35bccb7f25ab407382ed34af0b111475eb301c16f8a2e5/scipy-1.16.3-cp311-cp311-win_amd64.whl.metadata
     -------

In [1]:
# ==========================================================
# REAL-TIME TRAFFIC LIGHT DETECTION USING CNN + OPENCV
# ==========================================================

import cv2
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image

# -----------------------------
# DEVICE CONFIGURATION
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
#  DEFINE CNN MODEL (same as training)
# -----------------------------
class TrafficLightCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(TrafficLightCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# -----------------------------
#  LOAD TRAINED MODEL
# -----------------------------
model = TrafficLightCNN(num_classes=4).to(device)
model.load_state_dict(torch.load("traffic_light_cnn_best.pth", map_location=device))
model.eval()
print(" Model loaded successfully!")

# -----------------------------
#  IMAGE PREPROCESSING PIPELINE
# -----------------------------
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

class_names = ["Red Light", "Yellow Light", "Green Light", "Back of Lights"]

# -----------------------------
#  FUNCTION TO PREDICT FROM FRAME
# -----------------------------
def predict_from_frame(frame):
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    img_tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = model(img_tensor)
        probabilities = torch.softmax(outputs, dim=1)
        confidence, predicted = torch.max(probabilities, 1)
    return class_names[predicted.item()], confidence.item() * 100

# -----------------------------
#  START REAL-TIME DETECTION
# -----------------------------
cap = cv2.VideoCapture(0)  # Use 0 for webcam

if not cap.isOpened():
    print(" Error: Could not access camera.")
    exit()

print(" Real-time traffic light detection started. Press 'q' to exit.")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Frame not received. Exiting...")
            break

        label, confidence = predict_from_frame(frame)
        color = (0, 255, 0) if "Green" in label else (0, 255, 255) if "Yellow" in label else (0, 0, 255)
        text = f"{label} ({confidence:.1f}%)"

        cv2.putText(frame, text, (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
        cv2.rectangle(frame, (20, 20), (620, 460), (255, 255, 255), 2)

        cv2.imshow(" Traffic Light Detection", frame)

        # Properly detect 'q' press to quit
        key = cv2.waitKey(10) & 0xFF
        if key == ord('q') or key == 27:  # 'q' or ESC to quit
            print(" Exiting detection...")
            break

except KeyboardInterrupt:
    print("\n Interrupted manually. Closing...")

finally:
    cap.release()
    cv2.destroyAllWindows()
    print(" Camera released and windows closed successfully.")


 Model loaded successfully!
 Real-time traffic light detection started. Press 'q' to exit.
 Exiting detection...
 Camera released and windows closed successfully.
